# How to Train **YOLO11** Object Detection on a Custom Dataset
### (Drowsy Driver — 6 class) — theo đúng hướng dẫn Roboflow

YOLO11 cải tiến từ YOLOv9/v10: kiến trúc tốt hơn, C2PSA attention, ít tham số hơn YOLOv8 mà mAP cao hơn.

Dataset: `close_eyeL, close_eyeR, no_yawn, open_eyeL, open_eyeR, yawn`

`Runtime → Change runtime type → T4 GPU → Run all`

## Setup
### Cấu hình API key
Thử Colab Secrets (🔑 `ROBOFLOW_API_KEY`); nếu chưa set thì dùng key nhúng sẵn.

In [ ]:
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    assert ROBOFLOW_API_KEY
    print('✅  Dùng API key từ Colab Secrets')
except Exception:
    ROBOFLOW_API_KEY = 'qI3lEKlNpIZpNENdk3MH'    # ← key của bạn
    print('✅  Dùng API key nhúng sẵn')

### Before you start — kiểm tra GPU

In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print(HOME)
MODEL = 'yolo11s'    # đổi: yolo11n (nhẹ) / yolo11m / yolo11l (mAP cao hơn)

## Install YOLO11 via Ultralytics

In [ ]:
%pip install -q "ultralytics<=8.3.40" supervision roboflow
# không cho ultralytics theo dõi hoạt động
!yolo settings sync=False
import ultralytics
ultralytics.checks()

## Inference với model pre-trained trên COCO
### CLI

In [ ]:
!yolo task=detect mode=predict model={MODEL}.pt conf=0.25 source='https://media.roboflow.com/notebooks/examples/dog.jpeg' save=True

In [ ]:
from IPython.display import Image as IPyImage
IPyImage(filename=f'{HOME}/runs/detect/predict/dog.jpeg', width=600)

### SDK + supervision

In [ ]:
from ultralytics import YOLO
from PIL import Image
import requests
import supervision as sv

model = YOLO(f'{MODEL}.pt')
image = Image.open(requests.get('https://media.roboflow.com/notebooks/examples/dog.jpeg', stream=True).raw)
result = model.predict(image, conf=0.25)[0]
detections = sv.Detections.from_ultralytics(result)

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator(text_color=sv.Color.BLACK)
annotated_image = image.copy()
annotated_image = box_annotator.annotate(annotated_image, detections=detections)
annotated_image = label_annotator.annotate(annotated_image, detections=detections)
sv.plot_image(annotated_image, size=(10, 10))

## Fine-tune YOLO11 trên dataset custom
Tải dataset 6-class (format `yolov11`). Slug Roboflow **viết thường** — thử lần lượt tên.

In [ ]:
!mkdir -p {HOME}/datasets
%cd {HOME}/datasets

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = version = dataset = None
for proj in ['datio_yolo', 'driver-yawn', 'driver-yawn-wh6wj']:
    try:
        project = rf.workspace('nguyen-tuan-dat').project(proj)
        version = project.version(1)
        dataset = version.download('yolov11')
        print('✅  Dùng project:', proj)
        break
    except Exception:
        print('  ⏭️ ', proj, 'không tải được, thử tiếp...')
%cd {HOME}

In [ ]:
# Vá data.yaml nếu thiếu key train/val → xây lại từ thư mục (an toàn cho mọi format)
import yaml
from pathlib import Path
loc = Path(dataset.location)

old = {}
for yp in list(loc.glob('*.yaml')):
    with open(yp) as f: t = yaml.safe_load(f) or {}
    if 'names' in t: old = t; break
names = old.get('names')
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
if not names: names = ['close_eyeL','close_eyeR','no_yawn','open_eyeL','open_eyeR','yawn']

def imgdir(*c):
    for x in c:
        d = loc/x
        if d.exists() and any(d.iterdir()): return str(d.resolve())
    return None
cfg = {'train': imgdir('train/images','train'),
       'val':   imgdir('valid/images','valid','val') or imgdir('test/images','test'),
       'nc': len(names), 'names': names}
_t = imgdir('test/images','test')
if _t: cfg['test'] = _t
with open(f'{dataset.location}/data.yaml','w') as f: yaml.dump(cfg, f, sort_keys=False)
print('✅  data.yaml:', cfg)

## Custom Training

In [ ]:
%cd {HOME}
!yolo task=detect mode=train model={MODEL}.pt data={dataset.location}/data.yaml epochs=60 imgsz=640 batch=16 patience=20 plots=True

In [ ]:
import glob, os
TRAIN_DIR = max(glob.glob(f'{HOME}/runs/detect/train*'), key=os.path.getmtime)
BEST = f'{TRAIN_DIR}/weights/best.pt'
print('Train dir:', TRAIN_DIR)
!ls {TRAIN_DIR}

In [ ]:
from IPython.display import Image as IPyImage, display
for f in ['results.png','confusion_matrix.png','val_batch0_pred.jpg']:
    p = f'{TRAIN_DIR}/{f}'
    if os.path.exists(p): print(f); display(IPyImage(filename=p, width=620))

## Validate fine-tuned model

In [ ]:
!yolo task=detect mode=val model={BEST} data={dataset.location}/data.yaml

## Inference với model custom
### CLI

In [ ]:
!yolo task=detect mode=predict model={BEST} conf=0.25 source={dataset.location}/test/images save=True

In [ ]:
import glob
from IPython.display import Image as IPyImage, display
latest = max(glob.glob(f'{HOME}/runs/detect/predict*/'), key=os.path.getmtime)
for img in glob.glob(f'{latest}/*.jpg')[:4]:
    display(IPyImage(filename=img, width=500)); print('\n')

## Lưu weights về Google Drive (dùng cho file combine / Android)

In [ ]:
from google.colab import drive
import shutil, json
from pathlib import Path
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/DrowsyDriver_Results'); OUT.mkdir(parents=True, exist_ok=True)
shutil.copy(BEST, OUT/f'{MODEL}_best.pt')
(OUT/'summary_yolo11.json').write_text(json.dumps({'model':MODEL,'format':'yolov11','classes':names}, indent=2))
print('✅  Lưu Drive:', OUT/f'{MODEL}_best.pt')

## Deploy model lên Roboflow

In [ ]:
try:
    project.version(version.version).deploy(model_type='yolov11', model_path=f'{TRAIN_DIR}/')
    print('✅  Đã deploy lên Roboflow')
except Exception as e:
    print('⏭️  Deploy skip:', str(e)[:120])

## Chạy inference từ model hosted trên Roboflow

In [ ]:
!pip install -q inference
try:
    import os, random, cv2
    import supervision as sv
    from inference import get_model
    model_id = project.id.split('/')[1] + '/' + str(version.version)
    hosted = get_model(model_id, ROBOFLOW_API_KEY)
    test_loc = dataset.location + '/test/images/'
    for name in random.sample(os.listdir(test_loc), min(4, len(os.listdir(test_loc)))):
        img = cv2.imread(os.path.join(test_loc, name))
        res = hosted.infer(img, confidence=0.4, overlap=30)[0]
        det = sv.Detections.from_inference(res)
        ann = sv.BoxAnnotator().annotate(img.copy(), det)
        ann = sv.LabelAnnotator().annotate(ann, det)
        sv.plot_image(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB), size=(8,8))
except Exception as e:
    print('⏭️  Hosted inference skip (cần deploy xong ở cell trên):', str(e)[:120])

## 🏆 Hoàn tất
- `best.pt` đã lưu Drive (`yolo11s_best.pt`)
- Đã deploy lên Roboflow + chạy thử model hosted
- Bước tiếp: `colab_combine_6class.ipynb` để ensemble YOLO26 ⊕ YOLO11